In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')

In [3]:
import numpy as np
import pylab as plt

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

# Training

In [4]:
from vimms.Common import POSITIVE
from vimms.ChemicalSamplers import MZMLFormulaSampler, MZMLRTandIntensitySampler

from vimms_gym.env import DDAEnv
from vimms_gym.chemicals import generate_chemicals
from vimms_gym.features import obs_to_dfs
from vimms_gym.evaluation import evaluate
from vimms_gym.policy import fullscan_policy, random_policy, topN_policy, best_ppo_policy

warning in stationary: failed to import cython module: falling back to numpy
warning in coregionalize: failed to import cython module: falling back to numpy
warning in choleskies: failed to import cython module: falling back to numpy


In [5]:
# n_chemicals = (200, 500)
# mz_range = (100, 600)
# rt_range = (0, 300)
# intensity_range = (1E5, 1E10)

In [6]:
n_chemicals = (500, 2000)
mz_range = (100, 600)
rt_range = (200, 1000)
intensity_range = (1E5, 1E10)

In [7]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [8]:
isolation_window = 0.7
N = 10
rt_tol = 15
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE
noise_density = 0.3
noise_max_val = 1e4

In [9]:
mzml_filename = 'Beer_multibeers_1_fullscan1.mzML'
mz_sampler = MZMLFormulaSampler(mzml_filename, min_mz=min_mz, max_mz=max_mz)
ri_sampler = MZMLRTandIntensitySampler(mzml_filename, min_rt=min_rt, max_rt=max_rt,
                                       min_log_intensity=min_log_intensity,
                                       max_log_intensity=max_log_intensity)

2022-03-16 16:30:43.268 | DEBUG    | mass_spec_utils.data_import.mzml:_load_file:166 - Loaded 1109 scans
2022-03-16 16:30:47.201 | DEBUG    | mass_spec_utils.data_import.mzml:_load_file:166 - Loaded 1109 scans


In [10]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
    },
    'noise': {
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'N': N,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
        'min_ms1_intensity': min_ms1_intensity
    }
}

In [11]:
out_dir = 'results'
max_peaks = 20

total_timesteps = 5E6
n_eval_episodes = 10
deterministic = True

# Training

## DQN

In [12]:
model_name = 'DQN'

In [13]:
# default parameters
learning_rate = 0.0001
batch_size = 32
gamma = 0.99
exploration_fraction = 0.1
exploration_initial_eps = 1.0
exploration_final_eps = 0.05

# modified parameters
# learning_rate = 0.0003
# batch_size = 32
# gamma = 0.8
# exploration_fraction = 0.25
# exploration_initial_eps = 1.0
# exploration_final_eps = 0.10

In [14]:
env = DDAEnv(max_peaks, params)
check_env(env)

In [ ]:
model = DQN("MultiInputPolicy", env, 
            exploration_fraction=exploration_fraction, exploration_initial_eps=exploration_initial_eps, exploration_final_eps=exploration_final_eps, 
            tensorboard_log='./results/%s_DDAEnv_tensorboard' % model_name, 
            learning_rate=learning_rate, batch_size=batch_size, gamma=gamma)
model.learn(total_timesteps=total_timesteps)

In [16]:
# model = DQN("MultiInputPolicy", env, 
#             exploration_fraction=exploration_fraction, exploration_initial_eps=exploration_initial_eps, exploration_final_eps=exploration_final_eps, 
#             learning_rate=learning_rate, batch_size=batch_size, gamma=gamma, verbose=2)
# model.learn(total_timesteps=total_timesteps, log_interval=1)

In [17]:
mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=n_eval_episodes, 
                                          deterministic=deterministic)
mean_reward, std_reward

(243.57950590000002, 44.569410469477546)

In [18]:
fname = 'results/model_%s' % model_name
model.save(fname)

## PPO

In [19]:
model_name = 'PPO'

In [20]:
# default parameters
learning_rate = 0.0003
batch_size = 64
n_steps = 2048
ent_coef = 0.0
gamma = 0.99
gae_lambda = 0.95

# modified parameters
# learning_rate = 0.00001
# batch_size = 128
# n_steps = 256
# ent_coef = 0.01
# gamma = 0.1
# gae_lambda = 0.7

In [21]:
env = DDAEnv(max_peaks, params)
check_env(env)

In [22]:
model = PPO("MultiInputPolicy", env, 
            tensorboard_log='./results/%s_DDAEnv_tensorboard' % model_name, 
            learning_rate=learning_rate, batch_size=batch_size, n_steps=n_steps, 
            ent_coef=ent_coef, gamma=gamma, gae_lambda=gae_lambda)
model.learn(total_timesteps=total_timesteps)

In [23]:
# model = PPO("MultiInputPolicy", env, verbose=2, 
#             learning_rate=learning_rate, batch_size=batch_size, n_steps=n_steps, 
#             ent_coef=ent_coef, gamma=gamma, gae_lambda=gae_lambda)
# model.learn(total_timesteps=total_timesteps, log_interval=1)

In [24]:
mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=n_eval_episodes, 
                                          deterministic=deterministic)
mean_reward, std_reward

(260.005624, 32.88473949138659)

In [25]:
fname = 'results/model_%s' % model_name
model.save(fname)

# Controller test

Generate some chemical sets

In [26]:
chemical_creator_params = params['chemical_creator']

chem_list = []
for i in range(n_eval_episodes):
    print(i)
    chems = generate_chemicals(chemical_creator_params)
    chem_list.append(chems)

0
1
2
3
4
5
6
7
8
9


## Random controller

In [27]:
env = DDAEnv(max_peaks, params)
for i in range(n_eval_episodes):
    chems = chem_list[i]
    observation = env.reset(chems=chems)
    done = False
    num_steps = 0
    episode_reward = 0
    
    while not done:
        action = random_policy(observation)
        observation, reward, done, info = env.step(action)
        num_steps += 1
        episode_reward += reward
        if done:
            print(f'Episode {i} finished after {num_steps} timesteps with reward {episode_reward}')
            out_file = 'random_%d.mzML' % i
            env.write_mzML(out_dir, out_file) 
            print(evaluate(env))
            break

Episode 0 finished after 3663 timesteps with reward -2690.6605620553178
(0.532043530834341, 0.4725626673708675)
Episode 1 finished after 3660 timesteps with reward -2651.9496880174847
(0.5334128878281623, 0.460505734553568)
Episode 2 finished after 3655 timesteps with reward -2543.8047939418784
(0.47161572052401746, 0.3968957382486872)
Episode 3 finished after 3664 timesteps with reward -2618.0084248410785
(0.44950952106174263, 0.3738273521560024)
Episode 4 finished after 3683 timesteps with reward -2646.889891150746
(0.465979381443299, 0.39367095828513676)
Episode 5 finished after 3653 timesteps with reward -2727.8372967227974
(0.5969289827255279, 0.5419916899858048)
Episode 6 finished after 3647 timesteps with reward -2669.551280455446
(0.5249709639953543, 0.45653016063909513)
Episode 7 finished after 3644 timesteps with reward -2672.830482893106
(0.5098425196850394, 0.4551146004365098)
Episode 8 finished after 3654 timesteps with reward -2603.4386359507507
(0.4899665551839465, 0.418

## Top-N controller

In [28]:
N = 10
env = DDAEnv(max_peaks, params)
for i in range(n_eval_episodes):
    chems = chem_list[i]
    observation = env.reset(chems=chems)
    done = False
    num_steps = 0
    episode_reward = 0
    
    while not done:
        action = topN_policy(observation, N)
        observation, reward, done, info = env.step(action)
        num_steps += 1
        episode_reward += reward
        if done:
            print(f'Episode {i} finished after {num_steps} timesteps with reward {episode_reward}')
            out_file = 'top%d_%d.mzML' % (N, i)
            env.write_mzML(out_dir, out_file)
            print(evaluate(env))
            break

Episode 0 finished after 2333 timesteps with reward -58.57841322659519
(0.5477629987908101, 0.4447916132910473)
Episode 1 finished after 2331 timesteps with reward -30.487974356640283
(0.5525059665871122, 0.4324109401993659)
Episode 2 finished after 2351 timesteps with reward 39.96502093036854
(0.47762008733624456, 0.37365826651833156)
Episode 3 finished after 2350 timesteps with reward -7.126265593449716
(0.45470282746682056, 0.35791584590185294)
Episode 4 finished after 2341 timesteps with reward -2.1473482231249723
(0.4762886597938144, 0.3769951833635445)
Episode 5 finished after 2333 timesteps with reward -118.2964193397732
(0.6161228406909789, 0.5114697227019018)
Episode 6 finished after 2333 timesteps with reward -70.30028455618466
(0.5296167247386759, 0.42265933232528163)
Episode 7 finished after 2332 timesteps with reward -68.14586171653102
(0.5167322834645669, 0.42831372851644955)
Episode 8 finished after 2335 timesteps with reward -4.687181070364336
(0.49331103678929766, 0.39

## Fullscan controller

In [29]:
env = DDAEnv(max_peaks, params)
for i in range(n_eval_episodes):
    chems = chem_list[i]
    observation = env.reset(chems=chems)
    done = False
    num_steps = 0
    episode_reward = 0
    
    while not done:
        action = fullscan_policy(observation)
        observation, reward, done, info = env.step(action)
        num_steps += 1
        episode_reward += reward
        if done:
            print(f'Episode {i} finished after {num_steps} timesteps with reward {episode_reward}')
            out_file = 'fullscan_%d.mzML' % i
            env.write_mzML(out_dir, out_file)     
            print(evaluate(env))
            break

Episode 0 finished after 2001 timesteps with reward -199.99999999999292
(0.0, 0.0)
Episode 1 finished after 2001 timesteps with reward -199.99999999999292
(0.0, 0.0)
Episode 2 finished after 2001 timesteps with reward -199.99999999999292
(0.0, 0.0)
Episode 3 finished after 2001 timesteps with reward -199.99999999999292
(0.0, 0.0)
Episode 4 finished after 2001 timesteps with reward -199.99999999999292
(0.0, 0.0)
Episode 5 finished after 2001 timesteps with reward -199.99999999999292
(0.0, 0.0)
Episode 6 finished after 2001 timesteps with reward -199.99999999999292
(0.0, 0.0)
Episode 7 finished after 2001 timesteps with reward -199.99999999999292
(0.0, 0.0)
Episode 8 finished after 2001 timesteps with reward -199.99999999999292
(0.0, 0.0)
Episode 9 finished after 2001 timesteps with reward -199.99999999999292
(0.0, 0.0)


## Model-based Policy (DQN)

In [30]:
model_name = 'DQN'
fname = 'results/model_%s' % model_name
model = DQN.load(fname)

In [31]:
env = DDAEnv(max_peaks, params)
for i in range(n_eval_episodes):
    chems = chem_list[i]
    observation = env.reset(chems=chems)
    done = False
    num_steps = 0
    episode_reward = 0
    
    while not done:
        action, _states = model.predict(observation, deterministic=deterministic)
        observation, reward, done, info = env.step(action)
        num_steps += 1
        episode_reward += reward
        if done:
            print(f'Episode {i} finished after {num_steps} timesteps with reward {episode_reward}')
            out_file = '%s_%d.mzML' % (model_name, i)
            env.write_mzML(out_dir, out_file)
            print(evaluate(env))
            break

Episode 0 finished after 2218 timesteps with reward 212.78183876903464
(0.5102781136638452, 0.30814276067543933)
Episode 1 finished after 2216 timesteps with reward 226.99260055316154
(0.513126491646778, 0.3074010179872777)
Episode 2 finished after 2251 timesteps with reward 275.01564125161394
(0.4443231441048035, 0.2865400118611167)
Episode 3 finished after 2249 timesteps with reward 259.8558801728573
(0.4235429890363531, 0.2738624206420205)
Episode 4 finished after 2239 timesteps with reward 265.57910674034974
(0.44879725085910654, 0.29462754092384885)
Episode 5 finished after 2202 timesteps with reward 141.51586402846957
(0.5393474088291746, 0.3134487457477203)
Episode 6 finished after 2230 timesteps with reward 191.6334073200695
(0.5017421602787456, 0.30188337833583523)
Episode 7 finished after 2235 timesteps with reward 201.0987044771803
(0.5009842519685039, 0.2995541181758582)
Episode 8 finished after 2244 timesteps with reward 244.94906917551467
(0.47157190635451507, 0.288705950

## Model-based Policy (PPO)

In [32]:
model_name = 'PPO'
fname = 'results/model_%s' % model_name
model = PPO.load(fname)

In [33]:
env = DDAEnv(max_peaks, params)
for i in range(n_eval_episodes):
    chems = chem_list[i]
    observation = env.reset(chems=chems)
    done = False
    num_steps = 0
    episode_reward = 0
    
    while not done:
        action = best_ppo_policy(observation, model)
        observation, reward, done, info = env.step(action)
        num_steps += 1
        episode_reward += reward
        if done:
            print(f'Episode {i} finished after {num_steps} timesteps with reward {episode_reward}')
            out_file = '%s_%d.mzML' % (model_name, i)
            env.write_mzML(out_dir, out_file)
            print(evaluate(env))
            break

Episode 0 finished after 2231 timesteps with reward 223.28850043053757
(0.5175332527206772, 0.3003935512911146)
Episode 1 finished after 2216 timesteps with reward 242.30784247632943
(0.5178997613365155, 0.3002899753638726)
Episode 2 finished after 2254 timesteps with reward 295.5166138243565
(0.4546943231441048, 0.29167694429825414)
Episode 3 finished after 2249 timesteps with reward 273.9590904315027
(0.4362377380265436, 0.27606869659738237)
Episode 4 finished after 2243 timesteps with reward 264.19213954412743
(0.45017182130584193, 0.2920198190763194)
Episode 5 finished after 2212 timesteps with reward 135.37526493658638
(0.5009596928982726, 0.2723560473096199)
Episode 6 finished after 2227 timesteps with reward 202.69149513230983
(0.5063879210220673, 0.3019497844158708)
Episode 7 finished after 2229 timesteps with reward 204.5548759466315
(0.5019685039370079, 0.2949988938449466)
Episode 8 finished after 2255 timesteps with reward 262.7084737310911
(0.4807692307692308, 0.28979330782